In [8]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import warnings
import sys
sys.path.append('../')
from src import *

seed = 42
np.random.seed(seed)
warnings.simplefilter(action='ignore', category=FutureWarning)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

In [10]:
df = pd.read_parquet("../data/split/shuffled(test-1b).parquet")

In [4]:
df.iloc[1_000_000:].to_parquet(
            '../data/split/shuffled(test-1b).parquet',
            compression='zstd',      # лучшее сжатие
            index=False
        )

In [11]:
X, y = df[['X', 'Y', 'Wave', 'Intensity', 'group', 'center', 'brain']], df['label']
# df_shuffled.to_parquet(
#     '/home/noda/Projects/raman-spectroscopy/data/shuffled.parquet',
#     compression='zstd',      # лучшее сжатие
#     index=False
# )

In [12]:
del df
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
del X
del y

In [6]:
type(X_train)

pandas.DataFrame

In [13]:
from sklearn.metrics import f1_score
import joblib


model = CatBoostClassifier(
    iterations=1000,
    task_type="GPU",
    devices='0', # Индекс видеокарты
    loss_function='MultiClass', # У вас 3 класса
    early_stopping_rounds=50,
    verbose=100
)

model.fit(X_train, y_train, cat_features=['group', 'center', 'brain'])
joblib.dump(model, '../models/model-catboost.joblib')

y_pred = model.predict(X_test)

score = f1_score(y_test, y_pred, macro='average')

Learning rate set to 0.490922
0:	learn: 0.7868185	total: 5.75s	remaining: 1h 35m 42s
100:	learn: 0.0000122	total: 26.2s	remaining: 3m 52s
200:	learn: 0.0000011	total: 45.6s	remaining: 3m 1s
300:	learn: 0.0000005	total: 1m 4s	remaining: 2m 30s
400:	learn: 0.0000004	total: 1m 24s	remaining: 2m 5s
500:	learn: 0.0000003	total: 1m 43s	remaining: 1m 42s
600:	learn: 0.0000003	total: 2m 1s	remaining: 1m 20s
700:	learn: 0.0000002	total: 2m 20s	remaining: 1m
800:	learn: 0.0000002	total: 2m 40s	remaining: 39.8s
900:	learn: 0.0000002	total: 2m 59s	remaining: 19.7s
999:	learn: 0.0000002	total: 3m 17s	remaining: 0us


ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

TypeError: got an unexpected keyword argument 'macro'